# C2-linear-models — Session 3: Regularization and Sparsity

*One class session, roughly 85 minutes. Builds on Session 1 (the linear
model, residuals, MSE, and its gradient at given weights).*

**This session:** why the best training score can be a lie (C1's
overfitting, now with weights you can inspect); the **L2 penalty** and the
**L1 penalty** added to the MSE; the geometry of the two penalties —
circles versus diamonds, fading pull versus constant pull; the session's
centerpiece, a **complete Calc AB case analysis** of the one-dimensional
penalized minimum at the kink — the honest demonstration of *why L1 sets
weights exactly to zero while L2 only shrinks them*; **sparsity** and what
it buys; regularization paths read from supplied weight tables; a fully
worked concept MC and a fully worked constrained-coding example in the
exam registers; then unit-wide Exam Connections and Going Deeper.

Nothing in this session fits a model: every weight you'll see is given,
supplied, or derived in closed form by calculus.
Iterative weight-finding is `C3-gradient-descent`'s whole subject.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. When the Best Train Score Lies

**Motivation.**
Session 1 ended on a warning: the train column and the test column of
the scoreboard can disagree.
Here is that disagreement, engineered on purpose — and, because this is
a linear model, we can *read the weights* and see exactly what went
wrong.

**The setup.**
Seeded data with $d = 5$ features whose true recipe uses only the first
two: $y = 3x_0 - 2x_1 + 1 + \text{noise}$.
Only **12 training rows** (30 held out for testing) — few rows, many
knobs, the classic danger zone.
Two candidates, both *supplied* (the first was least-squares-fitted to
the 12 training rows elsewhere; how such fits are computed is C3's
business):

- $w^{\text{fit}} = (3.4817,\, -1.9717,\, -0.2093,\, 0.0327,\, -0.0649)$,
  $\;b^{\text{fit}} = 1.8853$ — tuned hard to the train rows;
- $w^{\text{plain}} = (3,\, -2,\, 0,\, 0,\, 0)$, $\;b^{\text{plain}} = 1$ —
  a modest guess matching the true recipe.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
X = rng.normal(0, 1, (42, 5))
y = 3 * X[:, 0] - 2 * X[:, 1] + 1 + rng.normal(0, 1.0, 42)
X_train, y_train = X[:12], y[:12]        # only 12 rows to learn from
X_test,  y_test  = X[12:], y[12:]


def predict(X, w, b):
    return (X * w).sum(axis=1) + b


def mse(X, y, w, b):
    return np.mean((predict(X, w, b) - y)**2)


w_fit = np.array([3.4817, -1.9717, -0.2093, 0.0327, -0.0649])   # supplied
b_fit = 1.8853
w_plain = np.array([3.0, -2.0, 0.0, 0.0, 0.0])
b_plain = 1.0

print(f"{'candidate':<12}{'train MSE':>12}{'test MSE':>12}")
print(f"{'fitted':<12}{mse(X_train, y_train, w_fit, b_fit):>12.4f}"
      f"{mse(X_test, y_test, w_fit, b_fit):>12.4f}")
print(f"{'plain':<12}{mse(X_train, y_train, w_plain, b_plain):>12.4f}"
      f"{mse(X_test, y_test, w_plain, b_plain):>12.4f}")

The fitted candidate crushes the train column ($0.40$ vs $1.70$) and
*loses* the test column ($1.61$ vs $1.15$): **overfitting**, C1's word,
now with an autopsy available.
Look at the fitted weights themselves:

- the three *irrelevant* features got weights
  $(-0.2093,\, 0.0327,\, -0.0649)$ — pure noise-chasing, since the true
  recipe ignores them;
- the informative weights are distorted ($3.48$ for $3$, $1.89$ for
  $1$) to help absorb the training noise.

With 12 rows and 6 knobs, the model has spare capacity, and spare
capacity gets spent memorizing noise.

**The remedy this unit teaches: charge for weights.**
Change the training-time objective from "make MSE small" to
$$L(w, b) \;=\; \mathrm{MSE}(w, b) \;+\; \text{penalty}(w),$$
where the penalty grows with the *size* of the weights.
Now noise-chasing has a price: a weight is only worth keeping if it buys
more MSE than it costs in penalty.
This design move is called **regularization**, and the two penalties
that matter — to this course and to the exam — are the subjects of the
next two sections.

### Checkpoint 1

1. In one sentence: why do *small* training sets make noise-chasing
   weights more likely? (Think knobs-per-row.)
2. Which three numbers inside $w^{\text{fit}}$ are the smoking gun for
   overfitting here, and what *should* they have been?
3. The penalty in $L$ depends on $w$ only — not on the data.
   In one sentence: what tension between the two terms does minimizing
   $L$ negotiate?

## 2. The L2 Penalty (Ridge)

**Definition.**
The **L2 penalty** charges the *squared size* of the weights:
$$\text{pen}_{L2}(w) \;=\; \lambda \sum_{k=0}^{d-1} w_k^2,
\qquad
L_{\text{ridge}}(w, b) \;=\; \mathrm{MSE}(w, b) + \lambda \sum_k w_k^2 ,$$
with **strength knob** $\lambda \ge 0$ set by you (not a weight — a
dial you choose; $\lambda = 0$ recovers plain MSE).
The combination is called **ridge** loss.

**The bias is not penalized.**
$b$ just sets the output's overall level to match the data's level; a
big $b$ is not model complexity, and charging for it would push
predictions systematically off-level (Pitfall 2 later makes this
concrete).
The sum runs over $w$ only — always.

**Evaluating at given weights.**
Penalties don't change *how* we evaluate — only *what*: the scoreboard
gains a column.
Section 1's two candidates, three choices of $\lambda$:

In [ ]:
def l2_penalty(w, lam):
    return lam * np.sum(w**2)


def ridge_loss(X, y, w, b, lam):
    return mse(X, y, w, b) + l2_penalty(w, lam)


print(f"{'lam':>6} {'cand':<8}{'train MSE':>11}{'penalty':>10}{'total L':>10}")
for lam in (0.0, 0.1, 1.0):
    for name, w_c, b_c in [("fitted", w_fit, b_fit), ("plain", w_plain, b_plain)]:
        print(f"{lam:>6} {name:<8}{mse(X_train, y_train, w_c, b_c):>11.4f}"
              f"{l2_penalty(w_c, lam):>10.4f}"
              f"{ridge_loss(X_train, y_train, w_c, b_c, lam):>10.4f}")

At $\lambda = 0$ the fitted candidate wins the total; by $\lambda = 1$
its bigger, noisier weights cost more than its MSE advantage, and the
plain candidate wins.
The knob $\lambda$ literally re-prices the trade-off.

**Differentiating at given weights.**
The penalty adds a term of its own (scan-the-sum, F4:
$\partial_{w_j} \sum_k w_k^2 = 2 w_j$):
$$\frac{\partial L_{\text{ridge}}}{\partial w_j}
   = \underbrace{\frac{2}{n}\sum_i r_i X_{ij}}_{\text{data term}}
     + \underbrace{2\lambda w_j}_{\text{penalty term}},
\qquad
\frac{\partial L_{\text{ridge}}}{\partial b}
   = \frac{2}{n}\sum_i r_i \quad (\text{unchanged}).$$
The penalty term $2\lambda w_j$ is a **pull toward zero with strength
proportional to $w_j$ itself**: strong on big weights, feeble on small
ones — remember that word *proportional*; it is the whole difference
from L1.

**Geometry.**
Level sets of $\sum_k w_k^2$ in the $(w_0, w_1)$ plane are **circles** —
the penalty only cares about distance from the origin, not direction.

In [ ]:
def ridge_grad_w(X, y, w, b, lam):
    r = predict(X, w, b) - y
    return (2 / X.shape[0]) * (r[:, None] * X).sum(axis=0) + 2 * lam * w


# central-difference check at given weights (checker MAY loop)
lam0 = 0.5
g = ridge_grad_w(X_train, y_train, w_fit, b_fit, lam0)
h, gap = 1e-6, 0.0
for j in range(5):
    step = np.zeros(5)
    step[j] = h
    est = (ridge_loss(X_train, y_train, w_fit + step, b_fit, lam0)
           - ridge_loss(X_train, y_train, w_fit - step, b_fit, lam0)) / (2 * h)
    gap = max(gap, abs(est - g[j]))
print("ridge grad:", np.round(g, 4))
print("check_gap :", gap)

# penalty geometry: circles vs (preview) diamonds
w0g, w1g = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
cs0 = axes[0].contour(w0g, w1g, w0g**2 + w1g**2, levels=[0.25, 1, 2.25, 4])
axes[0].clabel(cs0, fontsize=7)
axes[0].set_title(r"L2 level sets: $w_0^2 + w_1^2$ (circles)")
cs1 = axes[1].contour(w0g, w1g, np.abs(w0g) + np.abs(w1g),
                      levels=[0.5, 1, 1.5, 2])
axes[1].clabel(cs1, fontsize=7)
axes[1].set_title(r"L1 level sets: $|w_0| + |w_1|$ (diamonds)")
for ax in axes:
    ax.set_xlabel(r"$w_0$")
    ax.set_ylabel(r"$w_1$")
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Checkpoint 2

1. By hand: $w = (3, -1, 0)$, $\lambda = 0.5$.
   Compute $\text{pen}_{L2}$ and the penalty's contribution to each of
   the three partials.
2. Why is $b$ excluded from the penalty — one sentence.
3. As $w_j \to 0$, what happens to the L2 penalty's pull $2\lambda w_j$
   on that weight?
   (Answer now, remember in Section 4.)

## 3. The L1 Penalty and the Kink

**Definition.**
The **L1 penalty** charges *absolute* size:
$$\text{pen}_{L1}(w) \;=\; \lambda \sum_{k=0}^{d-1} |w_k|,
\qquad
L_{\text{lasso}}(w, b) \;=\; \mathrm{MSE}(w, b) + \lambda \sum_k |w_k| ,$$
(the combination is called **lasso** loss; bias again unpenalized).
Its level sets are the **diamonds** in the right panel above — corners
on the axes, exactly where weights are zero.
That corner is not decoration; it is the mechanism.

**The kink.**
Everything special about L1 comes from the graph of $|w|$: a V with a
**kink** at $0$.
Calc AB derivatives:

- for $w > 0$: $\dfrac{d}{dw}\lambda|w| = \lambda$;
- for $w < 0$: $\dfrac{d}{dw}\lambda|w| = -\lambda$;
- at $w = 0$: **no derivative exists** — the one-sided slopes disagree.

The one-sided slopes are all we need, and they are honest Calc AB
objects: approaching $0$ from the right the difference quotient tends to
$+\lambda$, from the left to $-\lambda$.
(If you meet the word **subgradient** later: it names exactly this
situation — at the kink, every slope between $-\lambda$ and $+\lambda$
fits under the V.
We will never need more than the two one-sided slopes.)

**Constant pull.**
Compare the two penalties' pull toward zero on a single weight:
$$\text{L2: } 2\lambda w \;(\text{fades as } w \to 0)
\qquad\text{vs}\qquad
\text{L1: } \lambda \cdot \mathrm{sign}(w) \;(\text{constant size } \lambda
\text{ all the way in}).$$
L1 charges the same toll per unit of weight no matter how small the
weight already is.
That non-fading toll is what will let it push weights *exactly* to zero
— demonstrated, not asserted, in the next section.

In [ ]:
# one-sided difference quotients of |w| at 0: the kink, measured
h = 1e-6
right = (abs(0 + h) - abs(0.0)) / h
left = (abs(0.0) - abs(0 - h)) / h
print("right slope:", right, "  left slope:", left)   # +1 vs -1

ws = np.linspace(-2, 2, 401)
plt.figure(figsize=(6.5, 3.6))
plt.plot(ws, np.abs(ws), label=r"$|w|$ — kink at 0")
plt.plot(ws, ws**2, "--", label=r"$w^2$ — smooth at 0")
plt.xlabel("w")
plt.title("The two penalties near zero: a corner vs a flat-bottomed bowl")
plt.legend()
plt.show()

### Checkpoint 3

1. By hand: $w = (2, -0.5, 0)$, $\lambda = 3$. Compute
   $\text{pen}_{L1}(w)$.
2. What are the two one-sided slopes of $\lambda|w|$ at $w = 0$ for
   $\lambda = 3$?
3. One sentence: near $w = 0$, which penalty's pull survives and which
   fades — and why does the fading one's bowl look "flat-bottomed" in
   the plot?

## 4. The One-Dimensional Argmin at the Kink, Fully Worked

**Motivation.**
"L1 makes weights exactly zero" is the exam's favorite regularization
fact.
We now *prove* it in the cleanest laboratory that still contains the
whole phenomenon: one weight, data pull captured by a single number.
Everything is Calculus AB — derivatives, case analysis, one-sided
slopes.
This closed-form minimization is the only weight-*finding* this unit
does, and it is done by calculus, not by iteration.

**The laboratory.**
$$f(w) \;=\; (w - a)^2 \;+\; \lambda\,|w| ,$$
where $a$ is where the data term alone would put the weight (its
unpenalized minimum) and $\lambda \ge 0$ is the penalty strength.
(Why this shape is the right laboratory: for a single standardized
feature, the MSE as a function of that one weight *is* a parabola with
its minimum at some $a$ — Session 1's loss, curvature rescaled; the
parabola stands in for any one weight's data pull.)

**The case analysis.**

*Step 0 — a minimum exists, and there are only two kinds of candidate.*
$f$ is continuous, and $f(w) \to +\infty$ as $|w| \to \infty$ (the
squared term dominates the linear penalty), so $f$ attains a global
minimum somewhere.
Away from $0$, $f$ is differentiable, so a minimizer with $w \ne 0$
must satisfy $f'(w) = 0$ (with a sign of $w$ consistent with its
region); the only other candidate is the kink $w = 0$.
We take the cases in turn.

*Case $w > 0$.*
Here $f(w) = (w - a)^2 + \lambda w$, so
$f'(w) = 2(w - a) + \lambda$, which vanishes at
$$w^\* = a - \frac{\lambda}{2}.$$
This candidate is only *admissible* if it actually lies in the region:
$a - \lambda/2 > 0$, i.e. $a > \lambda/2$.

*Case $w < 0$.*
Here $f(w) = (w - a)^2 - \lambda w$, so
$f'(w) = 2(w - a) - \lambda = 0$ at
$$w^\* = a + \frac{\lambda}{2},$$
admissible only if $a + \lambda/2 < 0$, i.e. $a < -\lambda/2$.

(On each smooth piece $f'' = 2 > 0$ — parabola plus straight line is
still an upward parabola — so an admissible critical point is that
piece's *global* minimum, never a maximum or a shelf.)

*Case $w = 0$ (the kink).*
The kink is a genuine minimum exactly when $f$ *increases in both
directions* from it — when the right-hand slope is $\ge 0$ and the
left-hand slope is $\le 0$:
$$f'(0^+) = 2(0 - a) + \lambda = \lambda - 2a \;\ge\; 0
   \iff a \le \tfrac{\lambda}{2},$$
$$f'(0^-) = 2(0 - a) - \lambda = -2a - \lambda \;\le\; 0
   \iff a \ge -\tfrac{\lambda}{2}.$$
Both hold exactly when $|a| \le \lambda/2$.
And the one-sided test is global, not merely local: on $w > 0$ the
slope $f'(w) = 2(w - a) + \lambda$ is *increasing* in $w$, so
$f'(0^+) \ge 0$ forces $f' \ge 0$ on the whole right half-line — $f$
never dips anywhere to the right of the kink; symmetrically on the
left.
When both slope conditions hold, the kink beats every other point
outright.

**Assembled result (the soft threshold).**
$$\boxed{\;w^\*(a, \lambda) \;=\;
\begin{cases}
a - \frac{\lambda}{2} & a > \frac{\lambda}{2}\\[2pt]
0 & |a| \le \frac{\lambda}{2}\\[2pt]
a + \frac{\lambda}{2} & a < -\frac{\lambda}{2}
\end{cases}\;}$$
The three cases tile all of $\mathbb{R}$ and meet continuously at
$|a| = \lambda/2$ (both formulas give $0$ there).
**This is the theorem the slogan compresses**: whenever the data pull is
weak enough — $|a| \le \lambda/2$ — the penalized minimum is *exactly*
$0$, not merely small.
And when the data pull wins, the answer is $a$ *moved toward zero by
$\lambda/2$*: L1 both selects and shrinks.

**The L2 comparison, same laboratory.**
$$g(w) = (w - a)^2 + \lambda w^2, \qquad
g'(w) = 2(w - a) + 2\lambda w = 0
\;\Longrightarrow\; \boxed{\;w^\* = \frac{a}{1 + \lambda}\;}$$
— no cases needed, $g$ is smooth.
The minimum is $a$ *divided* by $1 + \lambda$: shrunk toward zero,
**never equal to zero** unless $a = 0$ (at $w = 0$ the derivative is
$g'(0) = -2a \ne 0$, so $0$ is never even a critical point).
Proportional shrinkage cannot finish the job; a constant toll can.

**Worked numbers.**

| $a$ | $\lambda$ | L1 $w^\*$ | L2 $w^\*$ |
|---|---|---|---|
| $3$ | $4$ | $3 - 2 = 1$ | $3/5 = 0.6$ |
| $1.5$ | $4$ | $|1.5| \le 2 \Rightarrow \mathbf{0}$ | $1.5/5 = 0.3$ |
| $-0.9$ | $4$ | $\mathbf{0}$ | $-0.18$ |
| $-3$ | $2$ | $-3 + 1 = -2$ | $-1$ |

The code checks every closed form by brute evaluation on a dense grid —
*evaluation at (many) given weights*, this unit's licensed move, not
iteration:

In [ ]:
def f_l1(w, a, lam):
    return (w - a)**2 + lam * np.abs(w)


def soft_threshold(a, lam):
    """Closed-form argmin of (w - a)^2 + lam*|w| (the case analysis)."""
    if abs(a) <= lam / 2:
        return 0.0
    return a - np.sign(a) * lam / 2


grid = np.linspace(-4, 4, 800001)         # dense evaluation grid
print(f"{'a':>6}{'lam':>6}{'closed form':>13}{'grid argmin':>13}{'L2 a/(1+lam)':>14}")
for a, lam in [(3, 4), (1.5, 4), (-0.9, 4), (-3, 2)]:
    w_closed = soft_threshold(a, lam)
    w_grid = grid[np.argmin(f_l1(grid, a, lam))]
    print(f"{a:>6}{lam:>6}{w_closed:>13.4f}{w_grid:>13.4f}{a / (1 + lam):>14.4f}")

In [ ]:
# The two L1 regimes, drawn: interior minimum vs minimum AT the kink
ws = np.linspace(-2, 4, 500)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=False)
for ax, (a, lam) in zip(axes, [(3, 4), (1.5, 4)]):
    ax.plot(ws, f_l1(ws, a, lam), label=rf"$(w-{a})^2 + {lam}|w|$")
    w_star = soft_threshold(a, lam)
    ax.axvline(0, color="gray", linewidth=0.6)
    ax.plot([w_star], [f_l1(np.array([w_star]), a, lam)[0]], "o",
            label=rf"$w^* = {w_star}$")
    ax.set_xlabel("w")
    ax.legend(fontsize=8)
axes[0].set_title(r"data pull wins: $a > \lambda/2$")
axes[1].set_title(r"kink wins: $|a| \leq \lambda/2$ — exact zero")
plt.tight_layout()
plt.show()

**From one weight to many: the decoupled form.**
When the loss splits into a sum of independent one-weight problems,
$$F(w) \;=\; \sum_{k=0}^{d-1} \Big[ (w_k - a_k)^2 + \lambda |w_k| \Big],$$
each coordinate minimizes *its own* term, so the full argmin is the
soft threshold **applied coordinate-wise**:
$$w_k^\* = \begin{cases} 0 & |a_k| \le \lambda/2\\
a_k - \mathrm{sign}(a_k)\,\lambda/2 & \text{otherwise.}\end{cases}$$
(The general MSE does not decouple — its features interact — but the
decoupled form is exactly what happens with standardized, uncorrelated
features, and it is the exam's and p17's laboratory.)
Vectorized, no loops:

In [ ]:
a_vec = np.array([3.0, 1.5, -0.9, -3.0, 0.2])
lam = 4.0
w_star = np.where(np.abs(a_vec) <= lam / 2,
                  0.0,
                  a_vec - np.sign(a_vec) * lam / 2)
w_ridge = a_vec / (1 + lam)
print("a       :", a_vec)
print("L1  w*  :", w_star, "   exact zeros:", int((w_star == 0).sum()))
print("L2  w*  :", np.round(w_ridge, 4), "   exact zeros:", int((w_ridge == 0.0).sum()))

### Checkpoint 4

1. By hand, L1: compute $w^\*$ for $(a, \lambda) = (5, 4)$, for
   $(-1.8, 4)$, and for $(2, 8)$.
2. By hand, L2: same three pairs, $w^\* = a/(1+\lambda)$.
   Which of the six answers are exactly zero?
3. Boundary case: $a = \lambda/2$ exactly.
   Show that the case-$w>0$ formula and the kink case give the *same*
   $w^\*$, so the assembled formula is continuous there.
4. One sentence: in the case analysis, where *precisely* did the
   possibility of an exact zero come from — which mathematical feature
   of $|w|$ that $w^2$ lacks?

## 5. Why L1 Zeroes, Why L2 Shrinks — and What Sparsity Buys

**The force story (the theorem, retold).**
Stand at $w = 0$ and ask: is it worth stepping away?

- The *data term* $(w - a)^2$ offers a reward: its slope at $0$ is
  $-2a$, so a small step toward $a$ lowers it at rate $2|a|$.
- The *L1 toll* charges rate $\lambda$ for that same step — **constant**,
  no matter how small the step.
- Verdict: if $2|a| \le \lambda$, the toll eats the reward and the
  weight stays at exactly $0$ — which is precisely the case-analysis
  condition $|a| \le \lambda/2$.
- The *L2 toll* charges rate $2\lambda w \to 0$ near the origin: for
  small enough steps it is always cheaper than the reward, so the weight
  always steps off zero (when $a \neq 0$) — it just doesn't step far:
  $a/(1+\lambda)$.

**Definition (sparsity).**
A weight vector is **sparse** when many entries are *exactly* zero.
Zero weights are dropped features (Session 1, Checkpoint 1.3): the model
literally does not use them.
So L1 performs **automatic feature selection** while it fits the data —
and the surviving nonzero set tells you *which measurements mattered*.

What sparsity buys, concretely:

- **Interpretability** — "the model uses 4 of the 300 sensors" is a
  sentence a human can audit;
- **Cost** — unmeasured features never need collecting again (p15's
  scenario);
- **Speed and storage** — skipped multiplications.

**"Exactly zero" is a claim about arithmetic, not printing.**
An L2-shrunk weight of $10^{-3}$ *displays* as $0.00$ under rounding but
still uses its feature.
Testing sparsity means `w == 0` (for closed-form/supplied exact zeros),
never "looks small in the printout" — Pitfall 3 returns to this.

---

**Worked exam-style example I: concept MC (the r1 register).**

**Problem (reasoning is required; no code needed).**
A linear model is trained twice on the same data: once with loss
$\mathrm{MSE} + \lambda \sum_k w_k^2$, once with
$\mathrm{MSE} + \lambda \sum_k |w_k|$, both with large $\lambda$.
Which statement is correct?

A. The L2 run typically ends with many weights exactly zero, because
   squaring punishes small weights hardest.

B. The L1 run typically ends with many weights exactly zero, because the
   penalty's slope keeps constant magnitude $\lambda$ arbitrarily close
   to $w_k = 0$, so weak data pulls cannot hold a weight off zero.

C. Both penalties produce exactly zero weights equally often, since both
   are minimized at $w = 0$.

D. Neither penalty can produce exactly zero weights, since both are
   added to a smooth MSE.

E. The L1 run shrinks every weight by the same factor
   $1/(1 + \lambda)$.

**Solution.**
B states Section 4's theorem: near zero L1's pull is the constant
$\lambda$, and when $|a| \le \lambda/2$ (weak data pull) the exact zero
is the minimizer.
Eliminate A: squaring's pull $2\lambda w$ *vanishes* near zero — it
punishes *large* weights hardest, small ones barely at all.
Eliminate C: both penalties are minimized at $w = 0$, but the *sum*
data + penalty has its minimum at exactly $0$ only when the penalty's
slope can out-muscle the data slope there — which fading $2\lambda w$
cannot.
Eliminate D: smoothness of the MSE does not rescue differentiability of
$|w|$; the kink survives the sum, and kinks can be minima.
Eliminate E: the $1/(1+\lambda)$ factor is L2's closed form; L1
*subtracts* $\lambda/2$ (and clips to zero) — a shift, not a scaling.
**Answer B.**

---

### Checkpoint 5

1. Your colleague's L2-trained weights print as
   `[0.00, 1.31, -0.00, 0.42]` under two-decimal rounding.
   How many features does the model actually use, most likely — and what
   single line of code settles it?
2. In the force story, what plays the role of "$a$" for a real feature —
   what property of the data does it summarize?
3. One sentence: why is a sparse model cheaper *at data-collection
   time*, not just at compute time?

## 6. Regularization Paths from Supplied Weights

**Motivation.**
In practice you meet regularization as a *family* of trained models, one
per $\lambda$ — a **regularization path**.
Reading a path is an evaluation exercise, and evaluation is this unit's
home turf: every weight below is **supplied** (computed elsewhere; a
rounded snapshot per $\lambda$), and we do what this unit does — score,
count, plot, conclude.

Seeded data, true recipe $y = 3x_0 - 2x_1 + 1 + \text{noise}$ with 3
irrelevant features; 20 train / 10 test rows; bias fixed at $b = 1$ for
every supplied row.

In [ ]:
rng = np.random.default_rng(SEED)
X6 = rng.normal(0, 1, (30, 5))
y6 = 3 * X6[:, 0] - 2 * X6[:, 1] + 1 + rng.normal(0, 0.5, 30)
X6_tr, y6_tr, X6_te, y6_te = X6[:20], y6[:20], X6[20:], y6[20:]

lams = np.array([0.0, 0.1, 0.3, 1.0, 3.0, 10.0])

# SUPPLIED weight tables: row j is the trained w at lams[j] (given data).
W_l1 = np.array([[2.96, -1.98, 0.09, -0.12, 0.05],
                 [2.90, -1.92, 0.03, -0.05, 0.00],
                 [2.78, -1.80, 0.00,  0.00, 0.00],
                 [2.40, -1.45, 0.00,  0.00, 0.00],
                 [1.35, -0.50, 0.00,  0.00, 0.00],
                 [0.00,  0.00, 0.00,  0.00, 0.00]])
W_l2 = np.array([[2.96, -1.98, 0.09, -0.12, 0.05],
                 [2.88, -1.90, 0.08, -0.11, 0.04],
                 [2.70, -1.78, 0.07, -0.10, 0.04],
                 [2.28, -1.50, 0.05, -0.07, 0.03],
                 [1.55, -1.02, 0.03, -0.05, 0.02],
                 [0.74, -0.49, 0.01, -0.02, 0.01]])
b_path = 1.0


def path_mse(Xs, ys, W, b):
    """MSE of every supplied weight row at once: (rows, n) via broadcasting."""
    preds = (Xs[None, :, :] * W[:, None, :]).sum(axis=2) + b   # (6, n)
    return ((preds - ys)**2).mean(axis=1)                      # (6,)


zeros_l1 = (W_l1 == 0).sum(axis=1)
zeros_l2 = (W_l2 == 0).sum(axis=1)
print(f"{'lam':>6}{'L1 train':>10}{'L1 test':>9}{'L1 zeros':>9}"
      f"{'L2 train':>10}{'L2 test':>9}{'L2 zeros':>9}")
l1_tr, l1_te = path_mse(X6_tr, y6_tr, W_l1, b_path), path_mse(X6_te, y6_te, W_l1, b_path)
l2_tr, l2_te = path_mse(X6_tr, y6_tr, W_l2, b_path), path_mse(X6_te, y6_te, W_l2, b_path)
for j in range(6):
    print(f"{lams[j]:>6}{l1_tr[j]:>10.3f}{l1_te[j]:>9.3f}{zeros_l1[j]:>9}"
          f"{l2_tr[j]:>10.3f}{l2_te[j]:>9.3f}{zeros_l2[j]:>9}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, W, name in [(axes[0], W_l1, "L1 path"), (axes[1], W_l2, "L2 path")]:
    for k in range(5):
        ax.plot(lams, W[:, k], marker="o", markersize=3,
                label=rf"$w_{k}$" if k < 5 else None)
    ax.axhline(0, color="gray", linewidth=0.6)
    ax.set_xscale("symlog", linthresh=0.1)
    ax.set_xlabel(r"$\lambda$")
    ax.set_title(name)
axes[0].set_ylabel("weight value")
axes[0].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

**Reading the table and the picture.**

- **Zeros:** the L1 column marches $0 \to 1 \to 3 \to 5$ — features are
  *dropped*, irrelevant ones first (columns 2–4 die by
  $\lambda = 0.3$, exactly the noise features).
  The L2 column is $0$ everywhere: every weight survives, ever smaller.
  In the plot, L1 curves *hit the axis and stay*; L2 curves glide toward
  it and never land.
- **Train MSE** climbs as $\lambda$ grows — penalties trade training fit
  for smaller weights.
  (The tiny dip between the first two L1 rows is a reminder that these
  supplied weights are rounded snapshots, not exact minimizers.)
- **Test MSE:** here the gentle-$\lambda$ rows win — this dataset's 20
  training rows were enough that heavy regularization only hurts.
  When training data is scarcer (Section 1's 12-row regime), the test
  column dips at *intermediate* $\lambda$ before rising again; practice
  p14 hands you exactly such a path and asks you to find the dip.
  Either way the *method* is C1's: choose $\lambda$ by reading the
  held-out column, never the train column.

### Checkpoint 6

1. From the table: at which supplied $\lambda$ does the L1 path first
   use only the two informative features, and what is its zero-count
   there?
2. Why is counting `W_l2 == 0` giving zero everywhere *not* a bug in
   the count — which section's theorem predicted it?
3. One sentence: why must $\lambda$ be chosen with the test (held-out)
   column rather than the train column?

## 7. Worked Exam-Style Example II: Constrained Coding

The other exam register: implement penalized-loss evaluation under an
API ban and prove the gradient right numerically.
Solved step by step.

---

**Problem.**
Given seeded `X` of shape `(30, 4)`, targets `y` of shape `(30,)`,
weights `w` of shape `(4,)`, bias `b` (scalar), and `lam` (scalar), with
$$L(w, b) = \frac{1}{n}\sum_i \Big(\sum_k X_{ik} w_k + b - y_i\Big)^2
           + \lambda \sum_k w_k^2 ,$$
write `ridge_value_and_grad(X, y, w, b, lam)` returning the tuple
`(loss, grad_w, grad_b)` where `loss` is the scalar $L$, `grad_w` is the
shape-`(4,)` array of $\partial L/\partial w_j$, and `grad_b` is the
scalar $\partial L/\partial b$.
**Banned (zero points): `@`, `np.matmul`, `np.dot`, `.T`, `np.einsum`,
`np.inner`, `np.tensordot`, any `np.linalg` function, any loop.**
Then compute `check_gap`, the max abs difference of `(grad_w, grad_b)`
against central differences of your `loss` (the checker MAY loop).

---

**Solution.**

*Step 1 — formulas first.*
$r_i = \sum_k X_{ik} w_k + b - y_i$;
$L = \frac1n\sum r_i^2 + \lambda\sum_k w_k^2$;
$\partial L/\partial w_j = \frac{2}{n}\sum_i r_i X_{ij} + 2\lambda w_j$;
$\partial L/\partial b = \frac{2}{n}\sum_i r_i$.

*Step 2 — predictions without a banned product.*
`(X * w).sum(axis=1) + b`: broadcast $(30,4)\times(4,)$, collapse the
$k$-sum, shift — shape `(30,)`.

*Step 3 — loss.* `np.mean(r**2) + lam * np.sum(w**2)`; note `mean` for
the data term, `sum` for the penalty — mixing them up rescales one term
by $n$.

*Step 4 — gradients by broadcasting.*
`r[:, None] * X` is $(30, 1) \times (30, 4) \to (30, 4)$ — row $i$
scaled by $r_i$; `.sum(axis=0)` collapses the examples; scale by
$2/n$; add the penalty pull `2 * lam * w`.
The bias partial is the same route with an all-ones column:
`(2 / n) * r.sum()`.

*Step 5 — self-grade against the contract.*
Returns a tuple of (scalar, `(4,)`, scalar) ✓; no banned API, no loop
in the graded function ✓; the checker's loop is explicitly licensed ✓.

In [ ]:
rng = np.random.default_rng(SEED)
X7 = rng.normal(0, 1, (30, 4))
y7 = rng.normal(0, 1, 30)
w7 = np.array([0.8, -1.2, 0.0, 2.0])
b7 = -0.3
lam7 = 0.7


def ridge_value_and_grad(X, y, w, b, lam):
    n = X.shape[0]
    r = (X * w).sum(axis=1) + b - y                    # (n,)
    loss = np.mean(r**2) + lam * np.sum(w**2)
    grad_w = (2 / n) * (r[:, None] * X).sum(axis=0) + 2 * lam * w
    grad_b = (2 / n) * r.sum()
    return loss, grad_w, grad_b


loss7, gw7, gb7 = ridge_value_and_grad(X7, y7, w7, b7, lam7)
print("loss  :", loss7)
print("grad_w:", gw7, "  shape:", gw7.shape)
print("grad_b:", gb7)

In [ ]:
def L_only(w, b):
    return ridge_value_and_grad(X7, y7, w, b, lam7)[0]


h = 1e-6
check_gap = 0.0
for j in range(4):                                     # checker MAY loop
    step = np.zeros(4)
    step[j] = h
    est = (L_only(w7 + step, b7) - L_only(w7 - step, b7)) / (2 * h)
    check_gap = max(check_gap, abs(est - gw7[j]))
check_gap = max(check_gap,
                abs((L_only(w7, b7 + h) - L_only(w7, b7 - h)) / (2 * h) - gb7))
print("check_gap:", check_gap)                         # ~1e-9

### Checkpoint 7

1. The tempting one-liner for the data term of `grad_w` is
   `np.dot(r, X)`. Which ban does it hit, and what is the legal
   broadcasting rewrite?
2. Why does `grad_b` have *no* $2\lambda$ term?
3. Predict the three shapes: `r[:, None]`, `r[:, None] * X`, and the
   final `grad_w`.
   Then say which axis got summed and why.

## 8. Common Pitfalls II

**Pitfall 1 — reading the penalized total as "the error".**
The penalized loss is a *training-time objective*; predictive quality is
still measured by plain MSE on held-out data.
Comparing candidates trained at different $\lambda$ by their own
penalized totals mixes scoreboards — each candidate is being charged a
different tax — and the comparison can *invert* the truth.
Section 1's pair again: the fitted candidate came from a $\lambda = 0$
run (no tax at all), the plain candidate from a strong $\lambda = 1$ run
(its tax alone is $\lambda \sum_k w_k^2 = 9 + 4 = 13$):

In [ ]:
# Section 1's two supplied models, each with its own training-run lambda
lam_fit, lam_plain = 0.0, 1.0
tot_fit = mse(X_train, y_train, w_fit, b_fit) + lam_fit * np.sum(w_fit**2)
tot_plain = mse(X_train, y_train, w_plain, b_plain) + lam_plain * np.sum(w_plain**2)
print("penalized totals   : fitted", round(tot_fit, 4), " vs plain",
      round(tot_plain, 4), "  <- BROKEN comparison (different taxes)")
print("held-out plain MSEs: fitted", round(mse(X_test, y_test, w_fit, b_fit), 4),
      " vs plain", round(mse(X_test, y_test, w_plain, b_plain), 4),
      "  <- the real scoreboard")

The two scoreboards **disagree**: by penalized totals the fitted model
looks better by a factor of ~37 ($0.4006$ vs $14.7007$), yet on held-out
MSE it is the *worse* predictor ($1.6106$ vs $1.1453$).
The inversion is mechanical, not subtle: the plain candidate's total is
almost entirely tax ($13$ of its $14.7007$ is the $\lambda = 1$ penalty
charge), while the fitted candidate pays no tax at all — the teammate's
"loss" comparison measured tax brackets, not prediction.
Report the penalty-free test MSE when the question is "which model
predicts better"; the penalized total only ever referees weights
*within one* $\lambda$.

**Pitfall 2 — penalizing the bias.**
Add $b^2$ to the penalty and the objective starts rewarding
systematically-low predictions whenever the data's level is high — the
penalty picks a fight with the correct level and wins part of it:

In [ ]:
# Targets with a high level (true recipe needs b = 20)
rng = np.random.default_rng(SEED)
Xb = rng.normal(0, 1, (200, 2))
yb = 1.0 * Xb[:, 0] - 2.0 * Xb[:, 1] + 20.0 + rng.normal(0, 0.5, 200)
w_true, lam_b = np.array([1.0, -2.0]), 0.5

s = (Xb * w_true).sum(axis=1)              # the weights' contribution, fixed
b_closed = (yb - s).mean() / (1 + lam_b)   # Calc AB: set d/db to zero, solve

bs = np.linspace(5, 22, 851)               # wide enough to contain BOTH minima
correct = np.array([mse(Xb, yb, w_true, b) + lam_b * np.sum(w_true**2)
                    for b in bs])
broken = correct + lam_b * bs**2                    # BROKEN: + lam * b^2

print("mean(y - s)                    :", round((yb - s).mean(), 4))
print("best b, bias unpenalized (grid):", round(bs[np.argmin(correct)], 2))
print("best b, bias penalized (grid)  :", round(bs[np.argmin(broken)], 2))
print("closed-form penalized b        :", round(b_closed, 3))
print("MSE at the honest b            :", round(mse(Xb, yb, w_true, 20.0), 3))
print("MSE at the dragged-low b       :", round(mse(Xb, yb, w_true, b_closed), 2))

The broken objective is an upward parabola in $b$, so Calc AB solves it
exactly: $\frac{d}{db}$ of $\overline{(s + b - y)^2} + \lambda b^2$
vanishes at
$$b^\* = \frac{\overline{y - s}}{1 + \lambda}
       = \frac{20.0024}{1.5} \approx 13.335,$$
and the wide grid's argmin ($13.34$) agrees.
The penalty confiscates a third of the correct level — only
$1/(1+\lambda) = 2/3$ of it survives — so every prediction runs
$\approx 6.7$ low, and the MSE explodes from $0.237$ at the honest
$b = 20$ to $44.69$ at the dragged-low $b$.
The rule from Section 2 stands: the penalty sum runs over $w$ only.
(Two lab habits on display: the demo uses only evaluation at many given
$b$'s plus a closed form — no fitting; and the grid was deliberately
chosen wide enough to *contain* the closed-form answer.
A minimizer printed at a grid edge is a boundary artifact, not a
minimum — always check that your argmin lands strictly inside the
grid.)

**Pitfall 3 — "tiny" is not "zero", and the threshold is $\lambda/2$,
not $\lambda$.**
Two ways to fake sparsity by accident.
First, counting near-zeros as zeros — L2 rows *display* as zeros under
rounding while every feature stays in use.
Second, mis-remembering the soft threshold as "zero out when
$|a| \le \lambda$" — twice too eager:

In [ ]:
row = np.array([0.004, -0.002, 1.31, 0.0])     # an L2-ish weight row
print("printed at 2 decimals:", np.round(row, 2))
print("looks-like-zero count:", int((np.round(row, 2) == 0).sum()),
      " <- BROKEN sparsity claim")
print("exact-zero count     :", int((row == 0).sum()), " <- the truth")

a_vals = np.array([3.0, 1.8, -0.6])
lam = 2.5
wrong = np.where(np.abs(a_vals) <= lam, 0.0,
                 a_vals - np.sign(a_vals) * lam / 2)          # BROKEN: <= lam
right = np.where(np.abs(a_vals) <= lam / 2, 0.0,
                 a_vals - np.sign(a_vals) * lam / 2)
print("wrong threshold:", wrong, "  <- kills a = 1.8 wrongly")
print("right threshold:", right)                              # 1.8 survives

The theorem's boundary is $|a| \le \lambda/2$ — the factor comes from
the data parabola's slope $2(w - a)$, and forgetting the $2$ doubles the
kill zone.
When in doubt, re-run the one-line grid check from Section 4.

### Checkpoint 8

1. A teammate says "the $\lambda = 3$ model must be worse — its loss is
   $4.7$ against the $\lambda = 0.1$ model's $1.9$."
   Name the pitfall and the fair comparison.
2. Why does the mis-leveling from a penalized bias get *worse* as the
   targets' overall level grows?
3. With $\lambda = 5$: for which values of $a$ does the *correct* soft
   threshold give zero, and for which does the broken
   "$|a| \le \lambda$" version?

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **ML-concepts cluster** — 5 sub-parts, 50 points in r1-2026 —
  explicitly includes **regularization/sparsity** among its
  concept-MC territory (alongside supervised-vs-unsupervised,
  bias-variance, and metrics).
  These are five-option A–E items in exactly Section 5's register:
  *why* does L1 zero and L2 shrink, what does sparsity mean, which
  claim survives elimination.
  The analysis's difficulty profile counts the concept MCQs among the
  points reachable straight from the baseline plus units like this one —
  p03, p11, and p15 train the register.
- The **NumPy implementation cluster** (8 sub-parts, 55 points) features
  broadcasting-only gradient implementations with explicit API bans and
  zero-point clauses — Section 7's exact shape, inherited from F4.
  This unit adds the penalized-loss variants; p05–p10, p13, and p17
  drill them.
- The corpus style notes flag **numeric normal forms** (sign/gcd
  constraints making MC answers unique-decodable) — Session 1 §6's
  register, drilled by p04 — and the arc texture where **later parts
  consume earlier results** (a derivation feeds an implementation feeds
  a checker), which is the shape of p13, p14, and p17.
- Reasoning-required flags appear on derivation items; the 1-D argmin
  case analysis (Section 4, p11/p12) is this unit's derivation of
  record.

## Going Deeper

Optional forward pointers along the course map — nothing here is needed
for this unit's practice:

- **`C3-gradient-descent`** — the deferred half of this story.
  There the gradients you evaluated while standing still get *used*: the
  update $w \leftarrow w - \eta \nabla L$ (with $\eta$ the **learning
  rate**) walks downhill on exactly this unit's MSE and penalized
  losses, in exactly this unit's notation, and fits the linear model end
  to end.
- **`C5-neural-networks`** — what happens when the linear map's output
  is fed through nonlinearities and stacked: the model family grows, but
  the loss-plus-penalty design, the evaluate/differentiate discipline,
  and the sparsity vocabulary carry over unchanged.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. With few rows per knob, many weight settings fit the training rows
   almost equally well, and the tie is broken by whatever chases the
   sample's noise — nothing in the objective resists it.
2. The weights on the three irrelevant features,
   $(-0.2093, 0.0327, -0.0649)$ — the true recipe uses those features
   with weight exactly $0$.
3. Minimizing $L$ negotiates fit against size: a weight change is
   accepted only if it lowers the MSE by more than it raises the
   penalty.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $\text{pen}_{L2} = 0.5\,(9 + 1 + 0) = 5$; penalty contributions
   $2\lambda w_k = (3, -1, 0)$.
2. $b$ matches the data's overall level rather than adding model
   complexity, so charging for it just mis-levels predictions.
3. It fades to zero proportionally with $w_j$ — near the origin the L2
   penalty barely pulls at all.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $3\,(2 + 0.5 + 0) = 7.5$.
2. $+3$ from the right, $-3$ from the left.
3. L1's constant-size pull survives ($\pm\lambda$); L2's pull
   $2\lambda w$ fades — which is why $w^2$ looks flat-bottomed: its
   slope dies exactly where it would have been needed.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $(5, 4)$: $5 > 2 \Rightarrow w^\* = 5 - 2 = 3$.
   $(-1.8, 4)$: $|-1.8| \le 2 \Rightarrow w^\* = 0$.
   $(2, 8)$: $|2| \le 4 \Rightarrow w^\* = 0$.
2. $5/5 = 1$; $-1.8/5 = -0.36$; $2/9 \approx 0.222$.
   None are zero — only the two L1 kink-case answers were.
3. At $a = \lambda/2$: case formula gives
   $a - \lambda/2 = 0$ and the kink case gives $0$ — identical, so the
   assembled $w^\*$ is continuous at the boundary.
4. From the kink: $|w|$'s one-sided slopes at $0$ are $\pm 1$ (nonzero),
   so the penalty can hold the minimum *at* $0$ against a bounded data
   slope; $w^2$'s slope at $0$ is $0$ and can hold nothing.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Most likely all four (L2 zeros are exact only by coincidence):
   the printed $0.00$ and $-0.00$ are rounding.
   `int((w == 0).sum())` — or printing with full precision — settles it.
2. $a$ summarizes the feature's data pull — where the (one-weight,
   standardized) MSE parabola would put that weight with no penalty.
3. A weight that is exactly zero means the feature need never be
   measured again — the saving happens at the sensor, before any
   computation.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $\lambda = 0.3$: zeros $= 3$ (all three noise features gone, both
   informative features alive).
2. Section 4's L2 closed form $a/(1+\lambda)$ is never exactly zero for
   $a \ne 0$ — the count is faithfully reporting the theorem.
3. The train column rewards whatever fits the training rows —
   including their noise; only held-out rows measure prediction
   (C1's discipline, unchanged by penalties).

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. `np.dot` is on the banned list; rewrite as
   `(r[:, None] * X).sum(axis=0)`.
2. The penalty $\lambda\sum_k w_k^2$ contains no $b$, so its
   $b$-derivative is zero — the bias is unpenalized by design.
3. `(30, 1)`, `(30, 4)`, `(4,)`; axis 0 (the examples) is summed away
   because each partial $\partial L/\partial w_j$ aggregates every
   example's contribution to knob $j$.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1 — penalized totals at different $\lambda$ are different
   taxes; compare plain MSE on the same held-out rows.
2. The penalized optimum is $b^\* = \overline{y - s}/(1 + \lambda)$: a
   fixed *fraction* $\lambda/(1+\lambda)$ of the level is confiscated,
   so the absolute mis-leveling grows in proportion to the level
   itself.
3. Correct: $|a| \le 2.5$. Broken: $|a| \le 5$ — every $a$ with
   $2.5 < |a| \le 5$ is wrongly zeroed.

</details>